# 06 — DAGs and Automatic Differentiation
## From the chain rule to a tiny trainable neuron

**Prerequisites:** Python functions, derivatives of simple functions, and the idea of gradient descent.

### Learning goals

By the end of this notebook you should be able to:

- take a derivative through a small chain of operations **by hand**;
- see that same computation as a **graph** of little operations;
- explain how "the chain rule, done backwards" gives you every gradient in one pass;
- check a gradient numerically; and
- train a single neuron by rebuilding its graph on every step.

Every instructional example is complete and executable. Only the final project is intentionally unfinished.

### How to use this notebook

Run the cells from top to bottom. Every code cell is finished and runnable. Only the last cell (the project) is left for you.

You will see:

- **Predict** — before you run a cell, write down what you expect it to print, and why.
- **What you just saw** — a short note after a cell.
- **`assert` lines** — the specification. If one fails after you edit a cell, your change broke a rule.

In [1]:
from __future__ import annotations
import math, random
from typing import Callable

SEED = 7
random.seed(SEED)
print(f"Ready. Seed = {SEED}")

Ready. Seed = 7


## 1. The chain rule, by hand

Before writing any machinery, do by hand what the machinery will automate.

Take `y = (3x + 1)**2` at `x = 2`. Break it into two one-operation steps:

| step | value at `x = 2` | local derivative |
|---|---|---|
| `a = 3x + 1` | `a = 7` | `da/dx = 3` |
| `y = a**2` | `y = 49` | `dy/da = 2a = 14` |

The **chain rule**: the derivative of the whole is the **product** of the local derivatives along the path from `x` to `y`:

```
dy/dx = (dy/da) * (da/dx) = 14 * 3 = 42
```

Automatic differentiation does exactly this — but it starts at the output (`dy/dy = 1`) and multiplies the local derivatives **backwards** toward the inputs. One backward pass then gives the derivative with respect to *every* input at once.

**Predict** the four numbers the next cell prints: `a`, `y`, `dy_da`, `dy_dx`.

In [2]:
# FORWARD pass: compute each step and keep its value.
x = 2.0
a = 3 * x + 1          # step 1
y = a ** 2             # step 2
print(f"forward:  a = {a}   y = {y}")

# BACKWARD pass: start at dy/dy = 1, multiply by each local derivative going back to x.
dy_dy = 1.0
dy_da = dy_dy * (2 * a)     # local derivative of  y = a**2   is  2a
dy_dx = dy_da * 3           # local derivative of  a = 3x + 1  is  3
print(f"backward: dy/da = {dy_da}   dy/dx = {dy_dx}")

assert (a, y, dy_da, dy_dx) == (7.0, 49.0, 14.0, 42.0)

# Independent check: a "finite difference" nudges x a little each way and measures the slope.
# It never looks at the rule above, so it catches sign and chain-rule mistakes.
def finite_difference(f: Callable[[float], float], x: float, h: float = 1e-5) -> float:
    return (f(x + h) - f(x - h)) / (2 * h)

approx = finite_difference(lambda t: (3 * t + 1) ** 2, 2.0)
print(f"finite-difference estimate of dy/dx: {approx:.6f}   (matches 42)")
assert math.isclose(dy_dx, approx, rel_tol=1e-6)

forward:  a = 7.0   y = 49.0
backward: dy/da = 14.0   dy/dx = 42.0
finite-difference estimate of dy/dx: 42.000000   (matches 42)


### What you just saw

Two passes over the same three numbers:

- **Forward** computed each step and **kept** its value. The backward pass needs them — `dy/da` used `a`.
- **Backward** carried one running number (the gradient so far) from the output toward the input, multiplying by each step's local derivative.

For a straight-line computation, "backward mode" is just the chain rule written right to left. The rest of the notebook removes the "by hand" part: each operation will store its own local rule, and a graph walk will apply them in the right order — even when a value is used more than once.

## 2. A computation is a graph

If a value is used **more than once**, the computation is no longer a straight line — it is a graph.

A `Value` below is one node. `parents` are the nodes it was built from; `op` is the operation. In `z = x * y + x`, there are **two** paths from `x` to `z` (through the `*` and directly into the `+`), so both must add into `dz/dx`.

- "Directed" = information flows one way, inputs → output.
- "Acyclic" = no loops.
- Going backward, we visit each node **after** everything that used it, then apply its local rule.

![Computation graph](assets/autograd_dag.svg)

Each operation stores a small `_backward` function. For `u = a * b`: `da += b * du` and `db += a * du`. For `u = tanh(a)`: `da += (1 - tanh(a)**2) * du`. The `+=` (not `=`) is what lets several paths add up.

In [3]:
class Value:
    def __init__(self, data: float, parents=(), op: str = "", label: str = ""):
        self.data, self.grad = float(data), 0.0        # grad starts at 0
        self.parents, self.op, self.label = tuple(parents), op, label
        self._backward = lambda: None                  # filled in by each operation

    def __repr__(self): return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    __radd__ = __add__

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    __rmul__ = __mul__

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)

    def __pow__(self, exponent: float):
        out = Value(self.data ** exponent, (self,), f"**{exponent}")
        def _backward(): self.grad += exponent * self.data ** (exponent - 1) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data); out = Value(t, (self,), "tanh")
        def _backward(): self.grad += (1 - t*t) * out.grad
        out._backward = _backward
        return out

print("Value defined")

Value defined


### One node's local rule, on its own

Before walking a whole graph, watch a single `_backward`. Every operation returns a new `Value` whose `_backward` knows how to push gradient from that output back to its immediate parents — **if** `out.grad` is already set.

**Predict** `a.grad` and `b.grad` after the next cell sets `out.grad = 1.0` by hand and calls `out._backward()` once.

In [4]:
a = Value(3.0, label="a")
b = Value(-2.0, label="b")
out = a * b                              # out.data == -6.0

print("before: a.grad =", a.grad, " b.grad =", b.grad)

out.grad = 1.0                           # pretend `out` is the final result:  d out / d out = 1
out._backward()                          # run just this multiplication's rule

print("after : a.grad =", a.grad, " b.grad =", b.grad)
print("d(a*b)/da = b = -2 ;  d(a*b)/db = a = 3")
assert (a.grad, b.grad) == (-2.0, 3.0)

before: a.grad = 0.0  b.grad = 0.0
after : a.grad = -2.0  b.grad = 3.0
d(a*b)/da = b = -2 ;  d(a*b)/db = a = 3


## 3. Walking the whole graph backward

`_backward` on one node assumes that node's own `grad` is already finished. So for a full graph we must visit nodes in an order where each node comes **after** everything that used it.

The recipe:

1. set `root.grad = 1` (the derivative of the result with respect to itself),
2. visit the nodes in that "after everything that used me" order,
3. call each node's `_backward`.

Gradients use `+=` because a value can reach the output by several paths. For `z = x * y + x`, `dz/dx = y + 1`.

`backward` tracks which nodes it has already visited **by identity** (`id(node)`) — a node counts as "the same" only if it is literally the same object. Reusing one `Value` in two places is what makes the multi-path case, and identity tracking is what keeps its gradient a single running total.

**Predict** `x.grad` and `y.grad` after `backward(z)` for `z = x * y + x` with `x = 2`, `y = -3`.

In [5]:
def backward(root: Value) -> None:
    order, visited = [], set()
    def visit(node: Value):
        if id(node) in visited: return
        visited.add(id(node))
        for parent in node.parents:
            visit(parent)
        order.append(node)
    visit(root)
    root.grad = 1.0
    for node in reversed(order):
        node._backward()

x, y = Value(2, label="x"), Value(-3, label="y")
z = x * y + x
backward(z)

print(f"z.data = {z.data}   (2 * -3 + 2)")
print(f"x.grad = {x.grad}   (dz/dx = y + 1 = -3 + 1 = -2)")
print(f"y.grad = {y.grad}   (dz/dy = x = 2)")
assert (z.data, x.grad, y.grad) == (-4.0, -2.0, 2.0)

z.data = -4.0   (2 * -3 + 2)
x.grad = -2.0   (dz/dx = y + 1 = -3 + 1 = -2)
y.grad = 2.0   (dz/dy = x = 2)


### The visiting order, and why `+=` matters

The next cell prints the order `backward` used for `z = x * y + x`, then shows two ways gradients go wrong:

1. **A parameter reused across steps, with `grad` not reset.** In a training loop, the parameter nodes stay alive while the graph is rebuilt every step. Forget to set `grad = 0` and the next backward pass *adds* to the old value — the gradient comes out too big.
2. **Treating one reused value as two separate nodes.** Then each copy gets only one path's share, and neither holds the true `dz/dx`.

In [6]:
# 1. The order: parents first on the way in, so children first on the way back.
order, seen = [], set()
def _visit(node):
    if id(node) in seen: return
    seen.add(id(node))
    for p in node.parents: _visit(p)
    order.append(node)
_visit(z)
print("backward order:", [n.label or n.op or f"const({n.data:g})" for n in reversed(order)])

# 2. A parameter reused across two backward passes, grad NOT reset.
w = Value(1.5, label="w")
def build_loss():
    return (w * 2.0 - 1.0) ** 2          # (2w - 1)**2  ->  d/dw = 4*(2w - 1) = 8w - 4

backward(build_loss())
g1 = w.grad
print(f"first backward:  w.grad = {g1}   (correct: 8*1.5 - 4 = 8.0)")

backward(build_loss())                   # brand-new graph, but w.grad still holds 8.0
print(f"second backward (grad not reset): w.grad = {w.grad}   (wrong: it added another 8.0)")
assert math.isclose(w.grad, 2 * g1)

w.grad = 0.0                             # the fix: reset before each backward
backward(build_loss())
print(f"with w.grad = 0.0 first: w.grad = {w.grad}   (correct again)")
assert math.isclose(w.grad, g1)

# 3. Two separate nodes holding the same number are NOT one shared node.
xa, xb, yv = Value(2.0), Value(2.0), Value(-3.0)
backward(xa * yv + xb)
print(f"split graph: xa.grad = {xa.grad} (only the *y* path), xb.grad = {xb.grad} (only the +x path)")
print(f"their sum {xa.grad + xb.grad} equals the true dz/dx = y + 1, but neither node alone does")
assert xa.grad == -3.0 and xb.grad == 1.0

backward order: ['+', '*', 'y', 'x']
first backward:  w.grad = 8.0   (correct: 8*1.5 - 4 = 8.0)
second backward (grad not reset): w.grad = 16.0   (wrong: it added another 8.0)
with w.grad = 0.0 first: w.grad = 8.0   (correct again)
split graph: xa.grad = -3.0 (only the *y* path), xb.grad = 1.0 (only the +x path)
their sum -2.0 equals the true dz/dx = y + 1, but neither node alone does


## 4. Checking a gradient numerically

A **finite difference** estimates a derivative by nudging the input a little each way:

```
f'(x) ≈ (f(x + h) - f(x - h)) / (2h)
```

It is slower than backprop, but it does not use the backward code at all, so it catches sign errors and chain-rule slips. Pick a small but not tiny `h`: too large is inaccurate, too small loses precision to floating-point rounding.

The next cell checks the graph engine against `finite_difference` on three shapes: a composition, a reused node, and subtraction mixed with multiplication.

In [7]:
# Check 1: tanh(v**2).
v = Value(0.4); out = (v ** 2).tanh(); backward(out)
fd1 = finite_difference(lambda q: math.tanh(q * q), 0.4)
print(f"tanh(v**2)  : engine {v.grad:.6f}   finite-diff {fd1:.6f}")
assert math.isclose(v.grad, fd1, rel_tol=1e-5)

# Check 2: a*a + a  (a is reused, so its gradient must accumulate).  d/da = 2a + 1
a = Value(1.7); g = a * a + a; backward(g)
fd2 = finite_difference(lambda q: q * q + q, 1.7)
print(f"a*a + a     : engine {a.grad:.6f}   finite-diff {fd2:.6f}")
assert math.isclose(a.grad, fd2, rel_tol=1e-5)

# Check 3: (p - (-0.3)) * p.  d/dp = 2p + 0.3
p = Value(0.9); r = (p - Value(-0.3)) * p; backward(r)
fd3 = finite_difference(lambda t: (t + 0.3) * t, 0.9)
print(f"(p+0.3)*p   : engine {p.grad:.6f}   finite-diff {fd3:.6f}")
assert math.isclose(p.grad, fd3, rel_tol=1e-5)

print("all three gradient checks passed")

tanh(v**2)  : engine 0.779865   finite-diff 0.779865
a*a + a     : engine 4.400000   finite-diff 4.400000
(p+0.3)*p   : engine 2.100000   finite-diff 2.100000
all three gradient checks passed


## 5. One gradient-descent step, and why you reset gradients

Training repeats three parts:

1. **Rebuild** the graph for the current parameter values (fresh `Value` nodes).
2. **Backpropagate** the loss to fill in each `parameter.grad`.
3. **Update**: `parameter.data -= learning_rate * parameter.grad`.

Between steps you must set each `parameter.grad` back to `0`. The parameters themselves live on (they are what you are learning), but `.grad` is a running sum that belongs to **one** backward pass. Section 3 showed what a leftover gradient does.

**Predict** the two lines the next cell prints (loss and `w` after each step).

In [8]:
def loss_for(w_value: float) -> tuple[Value, Value]:
    """Tiny loss with its minimum at w = 3:  (w - 3)**2.  Returns (w_node, loss)."""
    w_node = Value(w_value, label="w")
    return w_node, (w_node - Value(3.0)) ** 2

lr = 0.1
w = 0.0
for step in range(2):
    w_node, loss = loss_for(w)
    backward(loss)
    grad = w_node.grad                       # d/dw (w - 3)**2 = 2(w - 3)
    w -= lr * grad
    print(f"step {step}: loss = {loss.data:6.3f}   grad = {grad:6.3f}   w -> {w:.3f}")

assert math.isclose(w, 1.08)                 # 0 -> 0.6 -> 1.08, heading toward 3
print("w is moving toward 3, where the loss is 0")
# loss_for() makes a fresh w_node each call, so there is no old .grad to clear here.
# The training loop below keeps ONE set of parameters alive, so it resets .grad itself.

step 0: loss =  9.000   grad = -6.000   w -> 0.600
step 1: loss =  5.760   grad = -4.800   w -> 1.080
w is moving toward 3, where the loss is 0


## 6. Training a neuron

A neuron computes `y_hat = tanh(w1*x1 + w2*x2 + b)`.

**Mean squared error** (MSE) measures how far predictions are from targets. One graph is built for the whole batch, then each parameter moves against its gradient: `param -= learning_rate * param.grad`.

The four data points below are separable by the sign of `x1`, so one neuron can fit them. The parameters `w1, w2, b` live for the whole loop, so the loop **resets their `.grad` every step** before `backward`. Watch the loss fall.

In [9]:
data = [((-1., -1.), -1.), ((-1., 1.), -1.), ((1., -1.), 1.), ((1., 1.), 1.)]
w1, w2, b = Value(.2, label="w1"), Value(-.1, label="w2"), Value(0., label="b")

def loss_value() -> Value:
    losses = []
    for (x1, x2), target in data:
        prediction = (w1 * x1 + w2 * x2 + b).tanh()
        losses.append((prediction - target) ** 2)
    return sum(losses) * (1 / len(losses))          # average = MSE

history = []
for _ in range(60):
    for parameter in (w1, w2, b):
        parameter.grad = 0.0                        # reset (section 5)
    loss = loss_value()
    backward(loss)
    for parameter in (w1, w2, b):
        parameter.data -= 0.1 * parameter.grad      # gradient-descent step
    history.append(loss.data)

print(f"loss: {history[0]:.4f}  ->  {history[-1]:.4f}   (dropped over 60 steps)")
print(f"learned: w1 = {w1.data:.3f}   w2 = {w2.data:.3f}   b = {b.data:.3f}")
print(f"w1 is large and positive (the target follows x1); w2 and b stayed near 0")
assert history[-1] < history[0]

loss: 0.6564  ->  0.0126   (dropped over 60 steps)
learned: w1 = 1.417   w2 = -0.003   b = -0.000
w1 is large and positive (the target follows x1); w2 and b stayed near 0


### Reusing this engine

`Value`, `backward` and `finite_difference` are also in the course's shared module, so the project can import them instead of copying the cells above:

```python
from course_utils import Value, backward, finite_difference
```

Same code, with docstrings. The cells above keep their own copy so this notebook still runs on its own.

## Project — Tiny multilayer perceptron

Extend the scalar engine into a small, configurable multilayer perceptron (MLP), then train it on XOR. XOR is useful here because one neuron cannot solve it: the hidden layer must combine multiple nonlinear units.

### Suggested milestones

1. Add `__truediv__`, `exp`, or ReLU, plus a `zero_grad()` helper.
2. Build a `Neuron` that computes `activation(sum(w_i * x_i) + b)` and a `Layer` that contains several neurons.
3. Build `MLP` from a list such as `[2, 3, 1]`, and expose its parameters for updates.
4. Train with MSE, rebuilding the graph on every iteration and clearing parameter gradients first.
5. Compare at least two analytic gradients with finite differences.

**Acceptance criteria:** shared-node gradients accumulate; gradient checks pass within a documented tolerance; XOR loss decreases substantially; the layer sizes are configurable; and old iteration graphs are not retained by the model or parameter containers.

**Checks to run yourself**

- Build `x * x` with one `Value`, backprop, and assert `grad == 2 * x` (accumulation across both parents).
- Gradient-check every new op (`exp`, `relu`, `/`) at two or three points against `finite_difference`.
- Train XOR from several random seeds; assert the final loss is below a threshold you pick and document.
- After training, hold a reference to one parameter and assert the previous iteration's graph is gone (`sys.getrefcount`, or check `parent` chains are not kept).
- Change `[2, 3, 1]` to `[2, 4, 4, 1]` and confirm only the constructor argument changed.

In [ ]:
# PROJECT WORKSPACE — intentionally incomplete
#
# Reuse the scalar engine from this notebook, or import it:
#     from course_utils import Value, backward, finite_difference
#
# 1. Add Value.exp / Value.relu / Value.__truediv__ and a zero_grad() helper.
# 2. Neuron(n_in), Layer(n_in, n_out), MLP(sizes) with a .parameters() list.
# 3. Train on XOR: [((0,0),0), ((0,1),1), ((1,0),1), ((1,1),0)] (or +/-1 targets),
#    rebuilding the graph and zeroing grads each step.
# 4. Gradient-check at least two parameters against finite_difference.

class MLP:
    ...


raise NotImplementedError("Implement and train the tiny MLP")
